# Selección del modelo

In [1]:
# Cargando las librerias necesarias
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import RFE, SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

# Customizaciones
pd.set_option('display.max_columns', None)

In [2]:
# Cargando los datos de la fase anterior de limpieza desde un fichero pickle
df = pd.read_pickle('./data/df_trans.pkl')

# Verificar el dataframe
print(df.head())
print(df.info())
print(df.shape)

   Diabetes_binary  HighBP  HighChol       BMI  Smoker  Stroke  \
0              0.0     1.0       0.0 -0.602885     0.0     0.0   
1              0.0     1.0       1.0 -0.602885     1.0     1.0   
2              0.0     0.0       0.0 -0.602885     0.0     0.0   
3              0.0     1.0       1.0 -0.281138     1.0     0.0   
4              0.0     0.0       0.0 -0.120264     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  NoDocbcCost   GenHlth  \
0                   0.0           1.0     0.0      1.0          0.0 -0.131141   
1                   0.0           0.0     1.0      0.0          0.0 -0.131141   
2                   0.0           1.0     1.0      1.0          0.0  1.681222   
3                   0.0           1.0     1.0      1.0          0.0 -0.131141   
4                   0.0           1.0     1.0      1.0          0.0  0.775041   

   MentHlth  PhysHlth  DiffWalk  Sex       Age  Education    Income  
0  0.148542  2.398501       0.0  1.0 -1.618374

## Hacer una selección de variables para reducir el input de los usuarios

In [3]:
X = df.drop(columns=["Diabetes_binary"])
y = df["Diabetes_binary"].astype(int)

# 1. Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X, y)
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

print("=== Importancia de variables (Random Forest) ===")
print(importancias)

# 2. Logistic Regression con L1 (Lasso)
log_l1 = LogisticRegression(penalty="l1", solver="liblinear", random_state=42)
log_l1.fit(X, y)
coefs = pd.Series(np.abs(log_l1.coef_[0]), index=X.columns).sort_values(ascending=False)

print("\n=== Importancia de variables (Logistic L1) ===")
print(coefs)

# 3. Selección de top 10 por RF
top10_rf = importancias.head(10).index.tolist()
# 4. Selección de top 10 por L1
top10_l1 = coefs.head(10).index.tolist()

# Intersección como "más estables"
selecionadas_RF_L1 = list(set(top10_rf) & set(top10_l1))

print("\nVariables recomendadas (intersección RF + L1):", selecionadas_RF_L1)

=== Importancia de variables (Random Forest) ===
BMI                     0.171385
Age                     0.130699
GenHlth                 0.104923
Income                  0.086991
HighBP                  0.076202
PhysHlth                0.071780
Education               0.060762
MentHlth                0.054858
HighChol                0.038955
Smoker                  0.030375
Fruits                  0.030187
Sex                     0.027146
DiffWalk                0.025544
PhysActivity            0.024499
Veggies                 0.023323
HeartDiseaseorAttack    0.019061
NoDocbcCost             0.012988
Stroke                  0.010324
dtype: float64

=== Importancia de variables (Logistic L1) ===
HighBP                  0.700934
GenHlth                 0.631376
HighChol                0.581645
BMI                     0.549436
Age                     0.458249
HeartDiseaseorAttack    0.269618
Sex                     0.250962
Stroke                  0.173322
Income                  0.1290

In [4]:
target = "Diabetes_binary"
X = df.drop(columns=[target])
y = df[target].astype(int)

# Escalado 0-1 SOLO para la selección (no modifica df)
X_nonneg = MinMaxScaler().fit_transform(X)

selector = SelectKBest(score_func=chi2, k=10)
selector.fit(X_nonneg, y)

selecionadas_KBest = X.columns[selector.get_support()].tolist()
print("Variables seleccionadas (chi²):", selecionadas_KBest)

Variables seleccionadas (chi²): ['HighBP', 'HighChol', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'GenHlth', 'PhysHlth', 'DiffWalk', 'Age']


## Hacer el split entre train y test

In [5]:
# Separar variables predictoras (X) y objetivo (y)
# Variables seleccionadas
# vars_seleccionadas = selecionadas_RF_L1
vars_seleccionadas = selecionadas_KBest
target = 'Diabetes_binary'

# Definir X e y solo con las variables seleccionadas
# X = df[vars_seleccionadas]
X = df.drop(columns=[target])
y = df[target].astype(int)

# Split en train y test (80/20) – sin estratificación (como indicaste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,stratify=y, random_state=99
)

print("Shape train:", X_train.shape, "Shape test:", X_test.shape)
print("\nDistribución target en train:\n", y_train.value_counts(normalize=True).round(3))
print("\nDistribución target en test:\n", y_test.value_counts(normalize=True).round(3))

Shape train: (54174, 18) Shape test: (13544, 18)

Distribución target en train:
 Diabetes_binary
1    0.508
0    0.492
Name: proportion, dtype: float64

Distribución target en test:
 Diabetes_binary
1    0.508
0    0.492
Name: proportion, dtype: float64


## Por Cross validation

In [6]:
# Separar variables predictoras (X) y objetivo (y)
# Variables seleccionadas
# vars_seleccionadas = selecionadas_RF_L1
# vars_seleccionadas = selecionadas_KBest
target = 'Diabetes_binary'

# Definir X e y solo con las variables seleccionadas
# X = df[vars_seleccionadas].copy()
X = df.drop(columns=[target])  # opción: todas las features
y = df[target].astype(int)
print("X shape:", X.shape, "| y shape:", y.shape)

X shape: (67718, 18) | y shape: (67718,)


## Preparar un pipeline para entrenar los modelos candidatos

In [7]:
# Definir pipelines de los modelos

pipelines = {
    "LogReg": Pipeline(steps=[
        ("model", LogisticRegression(max_iter=200, solver="lbfgs"))
    ]),
    "SVM-RBF": Pipeline(steps=[
        ("model", SVC(kernel="rbf", probability=True, random_state=42))  # datos ya escalados
    ]),
    "SVM-Linear": Pipeline(steps=[
        ("model", LinearSVC(random_state=42, max_iter=5000))
    ]),
    "RF": Pipeline(steps=[
        ("model", RandomForestClassifier(n_estimators=300, random_state=42))
    ]),
    "GB": Pipeline(steps=[
        ("model", GradientBoostingClassifier(random_state=42))
    ]),
}

## Crear la función para evaluar los modelos

In [8]:
# Función para evaluar modelos por split train/test

def evaluar_modelo_split(nombre, pipe, Xtr, ytr, Xte, yte, average="binary"):
    """
    Entrena y evalúa un pipeline en un split train/test.
    Devuelve un dict con métricas para poder rankear modelos.
    Criterio de ranking: AvgPrecision (PR-AUC) -> F1.
    """
    pipe.fit(Xtr, ytr)
    y_pred = pipe.predict(Xte)

    # Scores continuos para AUC/PR-AUC
    model_step = pipe.named_steps["model"]
    if hasattr(model_step, "predict_proba"):
        y_score = pipe.predict_proba(Xte)[:, 1]
    elif hasattr(model_step, "decision_function"):
        y_score = pipe.decision_function(Xte)
    else:
        y_score = None  # no hay score continuo

    # Métricas
    acc  = accuracy_score(yte, y_pred)
    prec = precision_score(yte, y_pred, average=average, zero_division=0)
    rec  = recall_score(yte, y_pred, average=average, zero_division=0)
    f1   = f1_score(yte, y_pred, average=average, zero_division=0)
    auc  = roc_auc_score(yte, y_score) if y_score is not None else np.nan
    ap   = average_precision_score(yte, y_score) if y_score is not None else np.nan  # PR-AUC

    print(f"\n=== {nombre} ===")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | ROC-AUC: {auc:.4f} | PR-AUC: {ap:.4f}")
    print(classification_report(yte, y_pred, digits=4))

    return {
        "Modelo": nombre,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC-AUC": auc,
        "AvgPrecision(PR-AUC)": ap
    }

In [9]:
# Función para evaluar modelos por cross-validation

def evaluar_modelo_cv(nombre, pipe, X, y, cv_splits=5, stratified=False, average="binary"):
    """
    Imprime classification_report por fold y devuelve métricas promedio.
    Selección posterior se hará por Average Precision (PR-AUC) y F1.
    """
    if stratified:
        from sklearn.model_selection import StratifiedKFold
        kf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
        splitter = kf.split(X, y)
    else:
        kf = KFold(n_splits=cv_splits, shuffle=True, random_state=42)
        splitter = kf.split(X)

    registros = []
    fold = 1
    for idx_tr, idx_te in splitter:
        X_tr, X_te = X.iloc[idx_tr], X.iloc[idx_te]
        y_tr, y_te = y.iloc[idx_tr], y.iloc[idx_te]

        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        # Scores continuos para AUC y Average Precision
        if hasattr(pipe.named_steps["model"], "predict_proba"):
            y_score = pipe.predict_proba(X_te)[:, 1]
        elif hasattr(pipe.named_steps["model"], "decision_function"):
            y_score = pipe.decision_function(X_te)
        else:
            y_score = None

        print(f"\n===== {nombre} | Fold {fold}/{cv_splits} =====")
        print(classification_report(y_te, y_pred, digits=4))

        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, average=average, zero_division=0)
        rec  = recall_score(y_te, y_pred, average=average, zero_division=0)
        f1   = f1_score(y_te, y_pred, average=average, zero_division=0)
        auc  = roc_auc_score(y_te, y_score) if y_score is not None else np.nan
        ap   = average_precision_score(y_te, y_score) if y_score is not None else np.nan  # PR-AUC

        registros.append({
            "fold": fold, "accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "roc_auc": auc, "avg_precision": ap
        })
        fold += 1

    df_cv = pd.DataFrame(registros)
    resumen = df_cv.mean(numeric_only=True).to_dict()
    print("\n===== Resumen CV (promedios) =====")
    print(pd.Series(resumen).round(4))
    return resumen, df_cv

## Entrenar y evaluar los modelos

In [10]:
# Entrenando y evaluando los modelos por split train/test

resultados_split = []

# Entrenando y evaluando los modelos por split train/test
for nombre, pipe in pipelines.items():
    m = evaluar_modelo_split(nombre, pipe, X_train, y_train, X_test, y_test, average="binary")
    resultados_split.append(m)

# Ranking según el mismo criterio que en CV: PR-AUC primero, F1 después
df_rank_split = pd.DataFrame(resultados_split).sort_values(
    by=["AvgPrecision(PR-AUC)", "F1"],
    ascending=False
).reset_index(drop=True)

print("\n===== Ranking (split train/test) — Prioridad: PR-AUC → F1 =====")
print(df_rank_split.round(4).to_string(index=False))

mejor_split = df_rank_split.iloc[0]["Modelo"]
print(f"\n🏆 Mejor modelo en split (criterio PR-AUC → F1): {mejor_split}")


=== LogReg ===
Accuracy: 0.7398 | Precision: 0.7353 | Recall: 0.7624 | F1: 0.7486 | ROC-AUC: 0.8147 | PR-AUC: 0.7952
              precision    recall  f1-score   support

           0     0.7449    0.7165    0.7304      6662
           1     0.7353    0.7624    0.7486      6882

    accuracy                         0.7398     13544
   macro avg     0.7401    0.7394    0.7395     13544
weighted avg     0.7400    0.7398    0.7396     13544


=== SVM-RBF ===
Accuracy: 0.7385 | Precision: 0.7169 | Recall: 0.8021 | F1: 0.7571 | ROC-AUC: 0.8044 | PR-AUC: 0.7642
              precision    recall  f1-score   support

           0     0.7669    0.6728    0.7168      6662
           1     0.7169    0.8021    0.7571      6882

    accuracy                         0.7385     13544
   macro avg     0.7419    0.7374    0.7369     13544
weighted avg     0.7415    0.7385    0.7373     13544


=== SVM-Linear ===
Accuracy: 0.7386 | Precision: 0.7317 | Recall: 0.7668 | F1: 0.7488 | ROC-AUC: 0.8147 | PR

In [11]:
# Entranamiento y evaluación por CV

resultados = []

# Entrenando y evaluando los modelos por CV
for nombre, pipe in pipelines.items():
    resumen, _df_folds = evaluar_modelo_cv(
        nombre, pipe, X, y,
        cv_splits=5, stratified=True, average="binary"  # pon stratified=True si lo prefieres
    )
    # Guardamos métricas clave para ranking
    resultados.append({
        "Modelo": nombre,
        "AvgPrecision(PR-AUC)": resumen.get("avg_precision", np.nan),
        "F1": resumen.get("f1", np.nan),
        "Precision": resumen.get("precision", np.nan),
        "Recall": resumen.get("recall", np.nan),
        "ROC-AUC": resumen.get("roc_auc", np.nan),
        "Accuracy": resumen.get("accuracy", np.nan),
    })

# Ranking de modelos (criterio: 1) AvgPrecision(PR-AUC), 2) F1)
df_rank = pd.DataFrame(resultados)
df_rank = df_rank.sort_values(
    by=["AvgPrecision(PR-AUC)", "F1"],
    ascending=False
).reset_index(drop=True)

print("\n===== Ranking de modelos (prioridad: PR-AUC, luego F1) =====")
print(df_rank.round(4).to_string(index=False))

mejor_modelo = df_rank.iloc[0]["Modelo"]
print(f"\n🏆 Mejor modelo por criterio PR-AUC → F1: {mejor_modelo}")


===== LogReg | Fold 1/5 =====
              precision    recall  f1-score   support

           0     0.7510    0.7079    0.7288      6662
           1     0.7321    0.7727    0.7519      6882

    accuracy                         0.7408     13544
   macro avg     0.7415    0.7403    0.7403     13544
weighted avg     0.7414    0.7408    0.7405     13544


===== LogReg | Fold 2/5 =====
              precision    recall  f1-score   support

           0     0.7514    0.7222    0.7365      6662
           1     0.7408    0.7687    0.7545      6882

    accuracy                         0.7458     13544
   macro avg     0.7461    0.7454    0.7455     13544
weighted avg     0.7460    0.7458    0.7456     13544


===== LogReg | Fold 3/5 =====
              precision    recall  f1-score   support

           0     0.7449    0.7137    0.7290      6662
           1     0.7337    0.7634    0.7483      6882

    accuracy                         0.7390     13544
   macro avg     0.7393    0.7386  